# TDM Data Upload — CDOT 2019 Loaded Network & TAZ

This notebook publishes the **CDOT 2019 Loaded Network** (line features) and the **CDOT 2019 TAZ** (polygon features) to the High Street ArcGIS Online organization as hosted feature layers.

## Source data

The two source files live under `data/TDM OD Matrices and Loaded Network/`:

| File | Geometry | Approx. size | Feature count |
|------|----------|--------------|---------------|
| `CDOT_2019_LoadNetwork.json` | LineString | ~419 MB | ~63,186 |
| `CDOT_2019_TAZ.json`         | Polygon    | ~51 MB   | ~8,045  |

These files are **not** plain GeoJSON. They are Cube/Voyager-style exports: a top-level metadata wrapper (`name`, `map_layer_type`, `bounds`, `center`, `zoom`, `property_names`, `type`, `features`). The `features` array itself is standard GeoJSON, so we can extract it into a `GeoDataFrame` once we strip the wrapper.

## What this notebook does

1. **Read** each TDM JSON file and convert it to a `geopandas.GeoDataFrame` in WGS84 (EPSG:4326).
2. **Sanitize** column names so they are valid ArcGIS field names (alphanumeric + underscore, no leading digit, no reserved characters, ≤ 64 chars).
3. **Authenticate** to ArcGIS Online via the `arcgis` Python API.
4. **Publish** each dataset as a hosted feature layer using the Spatially Enabled DataFrame (SeDF) `.spatial.to_featurelayer()` workflow, which handles chunked uploads for large data.
5. **Apply metadata** (title, tags, snippet, description) and set sharing.

## Prerequisites

Before running, make sure you have the required packages. From a terminal:

```text
conda install -c esri arcgis geopandas
```

or via pip (note: `arcgis` is significantly easier to install with conda):

```text
pip install arcgis geopandas pyogrio
```

You will also need:

- A **High Street ArcGIS Online** account with permission to publish hosted feature layers (Publisher role or higher).
- The org URL (for example `https://highstreet.maps.arcgis.com`).

Credentials are entered interactively at run time — they are **not** stored in this notebook.

---
## 1. Imports and configuration

Edit the values in the **CONFIG** cell below if file paths, item titles, or tags need to change. Everything downstream reads from this cell.

In [ ]:
import json
import re
import getpass
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape

from arcgis.gis import GIS
from arcgis.features import GeoAccessor, GeoSeriesAccessor  # noqa: F401  (registers the .spatial accessor on pandas)

print("geopandas :", gpd.__version__)
print("pandas    :", pd.__version__)

In [ ]:
# ============================================================
# CONFIG — edit values here, not below
# ============================================================

# Resolve paths relative to this notebook's location so it runs from anywhere.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
DATA_DIR  = REPO_ROOT / "data" / "TDM OD Matrices and Loaded Network"

# Source files (Cube/Voyager-wrapped GeoJSON).
TAZ_PATH     = DATA_DIR / "CDOT_2019_TAZ.json"
NETWORK_PATH = DATA_DIR / "CDOT_2019_LoadNetwork.json"

# ArcGIS Online destination.
AGOL_URL     = "https://highstreet.maps.arcgis.com"   # change if the org URL is different
AGOL_FOLDER  = "CDOT TDM 2019"                         # AGOL content folder; created if missing

# Item-level metadata applied to both layers.
COMMON_TAGS    = ["CDOT", "TDM", "2019", "Travel Demand Model", "Colorado"]
COMMON_LICENSE = "Source: Colorado Department of Transportation 2019 Statewide Travel Demand Model."

# Per-layer metadata.
LAYER_CONFIG = {
    "taz": {
        "src":   TAZ_PATH,
        "title": "CDOT 2019 TDM TAZ",
        "snippet": "CDOT 2019 statewide TDM Traffic Analysis Zones (8,045 zones).",
        "description": (
            "Traffic Analysis Zones (TAZ) from the Colorado Department of Transportation "
            "2019 Statewide Travel Demand Model. Includes households, population, employment, "
            "VMT, trips, and administrative attributes per zone."
        ),
        "tags": COMMON_TAGS + ["TAZ", "Zones"],
    },
    "network": {
        "src":   NETWORK_PATH,
        "title": "CDOT 2019 TDM Loaded Network",
        "snippet": "CDOT 2019 statewide TDM loaded highway network (~63K links).",
        "description": (
            "Loaded highway network from the Colorado Department of Transportation "
            "2019 Statewide Travel Demand Model. Includes link-level attributes for "
            "capacity, free-flow speed, daily flows, VMT, time-of-day speeds and times, "
            "and toll/HOV cost components."
        ),
        "tags": COMMON_TAGS + ["Loaded Network", "Highway", "Links"],
    },
}

# Sharing scope after publish: "private", "org", "public". Default is org-wide.
SHARE_SCOPE = "org"

# Sanity-check that the source files are where we expect.
for cfg in LAYER_CONFIG.values():
    assert cfg["src"].exists(), f"Missing source file: {cfg['src']}"
    print(f"OK  {cfg['src'].name:35s}  {cfg['src'].stat().st_size / 1e6:8.1f} MB")

---
## 2. Helpers — read TDM JSON, sanitize columns

### `read_tdm_geojson(path)`
The TDM JSON files have shape `{name, map_layer_type, bounds, ..., type: "FeatureCollection", features: [...]}`. We strip the wrapper, keep the `FeatureCollection` portion, and load it into a `GeoDataFrame`. The original Cube export uses lon/lat in WGS84, so we set the CRS explicitly to `EPSG:4326`.

### `sanitize_columns(gdf)`
ArcGIS field names must:

- contain only letters, digits, and underscores
- not start with a digit
- be at most 64 characters
- be unique within the layer

The TDM data uses names like `[Road Type]`, `[FACILITY TYPE]`, `[Shared3+]`, `[COUNTY_FCT_2020-2019]`. We strip brackets, replace anything that is not alphanumeric with `_`, collapse repeats, and de-duplicate. The original-name → cleaned-name mapping is returned so it can be saved alongside the layer for downstream users.

In [ ]:
def read_tdm_geojson(path: Path) -> gpd.GeoDataFrame:
    """Read a Cube/Voyager-wrapped GeoJSON file into a WGS84 GeoDataFrame.

    The full JSON is loaded into memory because the wrapper is a single object.
    For the ~419 MB Loaded Network this peaks around ~3-4 GB RAM during parse;
    if memory is tight, use ijson/streaming instead.
    """
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Wrapper sanity check.
    if raw.get("type") != "FeatureCollection" or "features" not in raw:
        raise ValueError(f"{path.name} is not a recognized TDM/GeoJSON FeatureCollection.")

    # The features list is already standard GeoJSON; geopandas can consume it directly.
    gdf = gpd.GeoDataFrame.from_features(raw["features"], crs="EPSG:4326")

    # Preserve the original feature id under an explicit column (geopandas drops it otherwise).
    if any("id" in feat for feat in raw["features"][:5]):
        gdf["orig_id"] = [feat.get("id") for feat in raw["features"]]

    print(f"  Read {len(gdf):,} features from {path.name}")
    print(f"  Geometry type(s): {gdf.geom_type.unique().tolist()}")
    print(f"  Columns: {len(gdf.columns)}")
    return gdf


_VALID = re.compile(r"[^A-Za-z0-9_]+")

def sanitize_columns(gdf: gpd.GeoDataFrame, max_len: int = 60) -> tuple[gpd.GeoDataFrame, dict]:
    """Rename columns to be ArcGIS-safe. Returns (renamed_gdf, original->clean mapping).

    Rules: alphanumeric + underscore only, no leading digit, max 60 chars
    (under the AGOL 64-char ceiling to leave room for de-duplication suffixes),
    and unique within the layer.
    """
    mapping: dict[str, str] = {}
    seen: set[str] = set()
    for col in gdf.columns:
        if col == "geometry":
            mapping[col] = col
            continue
        clean = _VALID.sub("_", str(col)).strip("_")
        if not clean:
            clean = "field"
        if clean[0].isdigit():
            clean = "f_" + clean
        clean = clean[:max_len]
        # Ensure uniqueness with a numeric suffix.
        base, n = clean, 1
        while clean.lower() in seen:
            suffix = f"_{n}"
            clean = base[: max_len - len(suffix)] + suffix
            n += 1
        seen.add(clean.lower())
        mapping[col] = clean

    renamed_count = sum(1 for k, v in mapping.items() if k != v)
    print(f"  Renamed {renamed_count} of {len(mapping)} columns")
    return gdf.rename(columns=mapping), mapping

---
## 3. Read and clean both datasets

We process the TAZ first (smaller; faster sanity check) and then the Loaded Network. The Loaded Network parse is the slow step — expect a few minutes.

In [ ]:
print("Reading TAZ ...")
taz_gdf = read_tdm_geojson(LAYER_CONFIG["taz"]["src"])
taz_gdf, taz_field_map = sanitize_columns(taz_gdf)
taz_gdf.head(2)

In [ ]:
print("Reading Loaded Network (this may take a few minutes) ...")
net_gdf = read_tdm_geojson(LAYER_CONFIG["network"]["src"])
net_gdf, net_field_map = sanitize_columns(net_gdf)
net_gdf.head(2)

### Quick spatial sanity check

Confirm both layers fall within Colorado (≈ lon −109 to −102, lat 37 to 41). If the bounds are wildly off, the source CRS assumption is wrong and we should fix that before publishing.

In [ ]:
print("TAZ bounds:    ", taz_gdf.total_bounds)
print("Network bounds:", net_gdf.total_bounds)

### (Optional) Save the field-name mapping as a CSV

Useful for downstream users who need to map the original Cube property names to the cleaned ArcGIS field names.

In [ ]:
out_dir = REPO_ROOT / "data" / "_field_maps"
out_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(taz_field_map.items(), columns=["original", "arcgis"]).to_csv(
    out_dir / "CDOT_2019_TAZ_fields.csv", index=False
)
pd.DataFrame(net_field_map.items(), columns=["original", "arcgis"]).to_csv(
    out_dir / "CDOT_2019_LoadNetwork_fields.csv", index=False
)
print("Field maps written to", out_dir)

---
## 4. Authenticate to ArcGIS Online

When this cell runs, you will be prompted for your AGOL **username** and **password**. Credentials are entered via `getpass` so the password is not echoed and is **not** saved in the notebook.

If your org uses SSO, swap the `GIS(...)` call for `GIS("home")` (when running inside ArcGIS Pro / AGOL Notebooks) or `GIS(AGOL_URL, client_id="...")` for OAuth. See the ArcGIS API for Python docs on authentication schemes.

In [ ]:
username = input("ArcGIS Online username: ")
password = getpass.getpass("ArcGIS Online password: ")

gis = GIS(AGOL_URL, username, password)
print(f"Signed in as {gis.users.me.username} ({gis.users.me.role}) on {gis.url}")

In [ ]:
# Make sure the destination folder exists. content.create_folder is a no-op
# if the folder already exists, so we look it up either way.
existing = [f for f in gis.users.me.folders if f["title"] == AGOL_FOLDER]
if not existing:
    gis.content.create_folder(AGOL_FOLDER)
    print(f"Created folder: {AGOL_FOLDER}")
else:
    print(f"Folder already exists: {AGOL_FOLDER}")

---
## 5. Publish helper

`publish_layer()` takes a cleaned `GeoDataFrame` plus the per-layer config dict and:

1. Converts the GDF to a Spatially Enabled DataFrame (SeDF).
2. Calls `.spatial.to_featurelayer()`, which creates a hosted feature layer and uploads features in chunks.
3. Updates the resulting item's metadata.
4. Sets sharing per `SHARE_SCOPE`.

If a layer with the same title already exists in `AGOL_FOLDER`, the function aborts with a message rather than silently creating a duplicate. To replace, delete the existing item first or change the title in `LAYER_CONFIG`.

In [ ]:
def publish_layer(gdf: gpd.GeoDataFrame, cfg: dict, gis: GIS, folder: str, share: str):
    """Publish a GeoDataFrame as a hosted feature layer; return the resulting Item."""
    title = cfg["title"]

    # Guard against accidental duplicates.
    owner = gis.users.me.username
    existing = gis.content.search(
        query=f'title:"{title}" AND owner:{owner}',
        item_type="Feature Layer",
    )
    if existing:
        raise RuntimeError(
            f"An item titled '{title}' already exists for {owner}: {existing[0].id}. "
            "Delete it or change the title in LAYER_CONFIG before re-running."
        )

    # Spatially Enabled DataFrame -> hosted feature layer. The arcgis API handles
    # chunked uploads internally, which is what makes the 63K-link network feasible.
    print(f"Publishing '{title}' ({len(gdf):,} features) ...")
    sedf = pd.DataFrame.spatial.from_geoframe(gdf)
    item = sedf.spatial.to_featurelayer(
        title=title,
        gis=gis,
        folder=folder,
        tags=cfg["tags"],
    )
    print(f"  Created item: {item.id}")

    # Apply the rest of the metadata after creation.
    item.update(
        item_properties={
            "snippet":      cfg["snippet"],
            "description":  cfg["description"],
            "licenseInfo":  COMMON_LICENSE,
            "accessInformation": "Colorado Department of Transportation (2019 Statewide TDM)",
        }
    )

    # Sharing.
    if share != "private":
        item.share(
            everyone=(share == "public"),
            org=(share in ("org", "public")),
        )
    print(f"  Shared: {share}")
    print(f"  URL:    {item.homepage}")
    return item

---
## 6. Publish the TAZ layer

Smaller of the two — runs in roughly a minute on a typical connection.

In [ ]:
taz_item = publish_layer(taz_gdf, LAYER_CONFIG["taz"], gis, AGOL_FOLDER, SHARE_SCOPE)

---
## 7. Publish the Loaded Network layer

This is the long step (large geometry payload + ~280 attribute columns × 63K rows). Plan on **10–30 minutes** depending on bandwidth. Don't kill the kernel; the API streams features in batches and resumes naturally on transient errors.

In [ ]:
net_item = publish_layer(net_gdf, LAYER_CONFIG["network"], gis, AGOL_FOLDER, SHARE_SCOPE)

---
## 8. Verify

Confirm both items exist, count features in the published service, and print direct links.

In [ ]:
for label, item in (("TAZ", taz_item), ("Loaded Network", net_item)):
    flayer = item.layers[0]
    count  = flayer.query(where="1=1", return_count_only=True)
    print(f"{label:14s}  id={item.id}  features={count:,}")
    print(f"               {item.homepage}")

---
## Troubleshooting

**`MemoryError` during JSON parse.** The Loaded Network is ~419 MB on disk and several GB in RAM once parsed. If you can't allocate that much, switch `read_tdm_geojson` to a streaming parse with `ijson` (parse `features.item` in a loop and append to a list).

**`HTTPError 503` / timeout during publish.** AGOL throttles large uploads. The arcgis API will retry, but if it gives up, re-run just the publish cell — `publish_layer` will fail fast on duplicate titles, so delete the partial item from My Content first.

**Field name collisions.** `sanitize_columns` only de-duplicates within a layer. If two original names sanitize to the same string (e.g. `[A B]` and `A_B`), the second gets a `_1` suffix. Inspect the CSV in `data/_field_maps/` to confirm.

**Replacing an existing layer.** `publish_layer` refuses to overwrite. To re-publish, either:

- delete the existing item in My Content and re-run, or
- change the `title` in `LAYER_CONFIG`, or
- switch to `FeatureLayerCollection.manager.overwrite(...)` against the existing item (preserves item id and any web maps that reference it).